In [1]:
!pip install gradio Sastrawi -q

import re
import joblib
import gradio as gr
import nltk
from nltk.tokenize import word_tokenize
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.7/209.7 kB 4.7 MB/s eta 0:00:00


True

In [5]:
from google.colab import files
print("Silakan upload svm_model.pkl dan tfidf_vectorizer.pkl:")
uploaded = files.upload()

Silakan upload svm_model.pkl dan tfidf_vectorizer.pkl:


Saving tfidf_vectorizer.pkl to tfidf_vectorizer.pkl
Saving svm_model.pkl to svm_model.pkl


In [6]:
model = joblib.load('svm_model.pkl')
tfidf = joblib.load('tfidf_vectorizer.pkl')
print("Model dan vectorizer berhasil dimuat!")
print("Kelas model:", model.classes_)

Model dan vectorizer berhasil dimuat!
Kelas model: ['negatif' 'positif']


In [7]:
stopword_factory = StopWordRemoverFactory()
stopwords = set(stopword_factory.get_stop_words())
stemmer = StemmerFactory().create_stemmer()

norm_dict = {
    "yg": "yang", "ga": "tidak", "gak": "tidak", "aja": "saja",
    "udah": "sudah", "banget": "sekali", "ok": "oke",
    "dki": "daerah khusus ibukota", "kalo": "kalau", "gk": "tidak",
    "apk": "aplikasi", "sy": "saya", "udh": "sudah", "tp": "tapi",
    "tdk": "tidak", "app": "aplikasi", "min": "admin", "pake": "pakai",
    "mulu": "selalu", "klo": "kalau", "utk": "untuk", "sdh": "sudah",
    "dah": "sudah", "eror": "error", "ko": "kok", "bgt": "sekali",
    "g": "tidak", "gw": "saya", "biar": "agar", "bs": "bisa",
    "tau": "tahu", "knp": "kenapa", "tgl": "tanggal", "blm": "belum",
    "ngisi": "mengisi", "doang": "saja", "bener": "benar", "ngga": "tidak",
    "sampe": "sampai", "karna": "karena", "gabisa": "tidak bisa",
    "dgn": "dengan", "sya": "saya", "trus": "terus", "kaya": "seperti",
    "sklrga": "keluarga", "baguss": "bagus"
}

def clean_text_bert(text):
    if not isinstance(text, str):
        return ''
    text = text.lower()
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    text = re.sub(r'@\w+|#\w+', '', text)
    text = re.sub(r'(.)\1{2,}', r'\1\1', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def normalize_text(text):
    text = str(text).lower()
    for word, replacement in norm_dict.items():
        text = re.sub(rf'\b{word}\b', replacement, text)
    return text

def preprocess_pipeline(text):
    text = clean_text_bert(text)
    text = normalize_text(text)
    tokens = word_tokenize(text)
    tokens = [t for t in tokens if t not in stopwords]
    text_stemmed = stemmer.stem(' '.join(tokens))
    return text_stemmed

In [8]:
def prediksi_sentimen(teks_input):
    try:
        teks_bersih = preprocess_pipeline(teks_input)
        vektor_teks = tfidf.transform([teks_bersih])

        prediksi = model.predict(vektor_teks)[0]
        probabilitas = model.predict_proba(vektor_teks)[0]
        kelas_model = model.classes_

        teks_detail = f"Teks setelah preprocessing: {teks_bersih}\n\nProbabilitas:\n"
        for i, kelas in enumerate(kelas_model):
            teks_detail += f"- {kelas}: {probabilitas[i]:.2%}\n"

        hasil_akhir = "🤩 POSITIF" if prediksi.lower() == 'positif' else "😡 NEGATIF"

        return hasil_akhir, teks_detail

    except Exception as e:
        return f"Error: {str(e)}", "Gagal memproses data."

# Uji cepat sebelum buka UI
print(prediksi_sentimen("Aplikasinya bagus sekali, sangat membantu"))

('🤩 POSITIF', 'Teks setelah preprocessing: aplikasi bagus sekali sangat bantu\n\nProbabilitas:\n- negatif: 1.08%\n- positif: 98.92%\n')


In [9]:
antarmuka = gr.Interface(
    fn=prediksi_sentimen,
    inputs=gr.Textbox(lines=4, placeholder="Masukkan ulasan di sini...", label="Input Ulasan"),
    outputs=[
        gr.Textbox(label="Prediksi Sentimen"),
        gr.Textbox(label="Detail & Probabilitas")
    ],
    title="Analisis Sentimen Aplikasi JAKI (SVM)",
    description="Model mendeteksi sentimen ulasan JAKI secara otomatis menggunakan algoritma Support Vector Machine, dilatih dengan label berdasarkan rating pengguna."
)

antarmuka.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://e5c8747162254f3c35.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
